# 엔진 RUL v12 — 8단계 파이프라인 · **CAP=125 버전** (체크포인트/재개)

`260630_v1_1`과 **독립 실행**되는 노트북입니다(이 파일 안에 모든 정의가 있음). 하나의 CNN이 ①RUL ②고장부위 ③열화시작 ④건강지표(θ)를 동시에 예측합니다.

**이 파일은 RUL 상한(CAP)=125 전용**입니다. CAP=50 / CAP=125 두 파일은 **체크포인트 폴더가 서로 달라**(`v12_ckpt_cap50` / `v12_ckpt_cap125`) 결과가 섞이지 않습니다. CAP이 다르면 RMSE 스케일도 다르니 두 버전의 RMSE를 직접 비교하지 마세요.

**핵심: fold별 체크포인트/재개.** 비싼 부분은 유닛 교차검증의 각 fold(모델 M개 학습)입니다. 그래서 **fold 하나가 끝날 때마다** 모델·예측을 드라이브에 저장하고, fold 안에서도 **모델 하나씩** 저장합니다. 다음 날 다시 켜면 끝난 fold는 **즉시 불러오고(학습 스킵)**, 안 끝난 fold(또는 안 끝난 모델)부터 이어서 합니다.

**8단계 지도**: 1 데이터통합+θ·hs → 2 θ라벨 → 3 regime정규화+잔차 → 4 멀티헤드CNN → 5 불확실성+딥앙상블 → 6 고장분류살리기 → 7 conformal → 8 **체크포인트 실행+저장**. 각 코드에 `[N단계]` 주석 표시.

**θ는 시뮬 정답**이라 실엔진엔 없습니다(θ 기반 기능은 추정 능력 시연).

**실행**: 위→아래로 정의 셀 전부 실행 → `prepare()` → fold 루프(이미 끝난 fold 즉시 스킵) → `finalize()`.

## 설정 (CONFIG)

모든 손잡이가 여기 모여 있습니다. 데이터 폴더는 자동 탐색하고, 결과는 드라이브(`RESULT_DIR`)에 저장해 세션이 꺼져도 남게 했습니다.

핵심 값의 의미:
- `CAP` = RUL 상한(그 위는 다 상한으로 자름). 우리는 125.
- `L / DOWN / STRIDE` = 윈도우 길이 / 다운샘플 / 윈도우 간격. 우리 검증값(50·10)에 v12의 낮은 겹침(STRIDE=20)을 결합했습니다.
- `K_FOLDS` = 유닛(엔진) 교차검증 폴드 수. `ENSEMBLE_M` = fold마다 학습할 모델 수(딥앙상블).
- `LAMBDA_*` = 멀티태스크에서 각 작업 손실의 비중. `FAULT_CLASS_WEIGHT` = 고장분류 붕괴 방지 스위치(6단계 핵심).

In [ ]:
# ===== 설정 (CONFIG) — CAP=125 버전 =====
try:
    from google.colab import drive; drive.mount('/content/drive')
    import glob as _glob, os as _os
    _hits = _glob.glob('/content/drive/MyDrive/**/N-CMAPSS_DS*.h5', recursive=True)
    DATA_DIR = _os.path.dirname(_hits[0]) if _hits else "/content/drive/MyDrive/제주항공"
except Exception:
    DATA_DIR = "."

CAP = 125          # ★ 이 파일의 RUL 상한 (이 파일은 125 전용)
CKPT_DIR   = f"{DATA_DIR}/v12_ckpt_cap{CAP}"     # fold별 체크포인트(모델·예측)
RESULT_DIR = f"{DATA_DIR}/v12_결과_cap{CAP}"      # 최종 JSON
SUBSETS = []                                      # 비우면 모든 N-CMAPSS_DS*.h5 자동

L = 50; DOWN = 10; STRIDE = 20; MAX_WIN = 120000
EPOCHS = 30; BATCH = 256; LR = 1e-3; WD = 1e-2
K_FOLDS = 6; ENSEMBLE_M = 3; WARMUP_MSE = 5; N_REGIMES = 8
USE_REGIME_NORM = True; USE_RESIDUAL = True; USE_HIER = False
ALPHA = 0.10; GROUP_CONDITIONAL = True; MIN_CAL_PER_GROUP = 50
MULTITASK = True; LAMBDA_FAULT = 0.3; LAMBDA_HS = 0.3; LAMBDA_THETA = 0.5
TERM_T = 0.0; FAULT_CLASS_WEIGHT = True
SUBSET_OFFSET = 1000; TEST_OFFSET = 50000; SEED0 = 0

import h5py, numpy as np, json, time, os, pickle
import torch, torch.nn as nn, torch.nn.functional as Fn
from torch.utils.data import TensorDataset, DataLoader
try:
    from sklearn.cluster import MiniBatchKMeans; HAVE_SK = True
except Exception:
    HAVE_SK = False
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
FCN = {1: '단거리', 2: '중거리', 3: '장거리'}
MODULES = ['Fan', 'LPC', 'HPC', 'HPT', 'LPT']; NMOD = len(MODULES)
HAS_THETA = True
rng = np.random.RandomState(0)
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(RESULT_DIR, exist_ok=True)
print("장치:", dev, "| 데이터:", DATA_DIR)
print(f"CAP={CAP} | 체크포인트 폴더: {CKPT_DIR}")

## 1단계 — 데이터 통합 + θ·hs 읽기

**목표**: 흩어진 N-CMAPSS 파일들을 하나로 합치고, 기존에 안 읽던 **θ(각 모듈 건강파라미터)** 와 **hs(열화상태)** 까지 읽어옵니다. 뒤의 모든 단계(고장·건강)가 이 데이터 위에서 돌아가므로 토대입니다.

**핵심 함수**
- `_load(path, split)` — 파일 하나에서 한 split(dev/test)을 읽습니다. `X_s`(센서), `W`(운항조건), `Y`(RUL), `A`(메타: 유닛·사이클·비행클래스·hs), 그리고 **`T`(θ)** 를 읽어요.
- `_localize(path)` — 드라이브의 큰 h5를 `/content`로 복사해 읽기 속도를 높입니다(선택적 최적화).
- `load_multi()` — 모든 파일 × (dev/test) 를 합칩니다. **유닛 번호에 오프셋**(`+파일i×1000`, test는 `+50000`)을 더해 서로 다른 엔진이 안 겹치게 합니다. 마지막에 2단계 함수(`_fault_per_unit`, `_theta_dev`)를 호출해 라벨까지 붙여 돌려줍니다.

**현재 코드와의 차이**: 기존 `load_file`은 dev만, 5개 필드(W·Xs·Y·유닛·비행클래스)만 읽고 클래스별로 따로 처리했어요. 여기선 **모든 파일+split을 통합**하고 **θ·hs·사이클**까지 읽어 하나의 풀(pool)로 다룹니다.

In [ ]:
# ===== [1단계] 로딩 + 통합 =====
def _load(path, split):
    """파일 1개의 한 split(dev/test) 읽기. θ(T)·hs 포함."""
    with h5py.File(path, 'r') as f:
        if 'X_s_' + split not in f: return None
        Xs = f['X_s_' + split][:].astype(np.float32); W = f['W_' + split][:].astype(np.float32)
        Y = f['Y_' + split][:].astype(np.float32).ravel(); A = f['A_' + split][:]
        T = f['T_' + split][:].astype(np.float32) if 'T_' + split in f else None
    unit = A[:, 0].astype(int); cyc = A[:, 1].astype(int); Fc = A[:, 2].astype(int)
    hs = A[:, 3].astype(int) if A.shape[1] > 3 else np.ones(len(unit), int)
    return Xs, W, Y, unit, cyc, Fc, hs, T

def _localize(path):
    """드라이브 큰 파일을 /content로 복사(읽기 속도↑). 실패해도 원본 경로 반환."""
    import shutil
    if not path.startswith('/content/drive'): return path
    local = '/content/' + os.path.basename(path)
    if os.path.exists(local) and os.path.getsize(local) > 0: return local
    for att in range(3):
        try:
            print(f"  [복사] {os.path.basename(path)} -> /content (시도 {att+1}) ..."); shutil.copy(path, local); return local
        except Exception as e:
            print(f"    복사 실패: {e}")
            if os.path.exists(local):
                try: os.remove(local)
                except Exception: pass
    raise RuntimeError(f"복사 3회 실패: {path}")

def load_multi():
    """모든 DS + dev/test를 하나로 합침. 유닛 오프셋으로 충돌 방지. 라벨(2단계)까지 부착."""
    global HAS_THETA
    import glob as _glob
    subs = SUBSETS if SUBSETS else sorted(os.path.basename(p) for p in _glob.glob(os.path.join(DATA_DIR, "N-CMAPSS_DS*.h5")))
    blocks = []; nsens = None; used = []
    for i, ds in enumerate(subs):
        dpath = f"{DATA_DIR}/{ds}"
        if not os.path.exists(dpath): print(f"  [없음] {ds}"); continue
        try: path = _localize(dpath)
        except Exception as e: print(f"  [건너뜀] {ds}: {e}"); continue
        for s in ('dev', 'test'):
            try: d = _load(path, s)
            except Exception as e: print(f"  [읽기실패] {ds}/{s}: {e}"); continue
            if d is None: continue
            if nsens is None: nsens = d[0].shape[1]
            if d[0].shape[1] != nsens: print(f"  [skip] {ds}/{s}"); continue
            off = i * SUBSET_OFFSET + (TEST_OFFSET if s == 'test' else 0)
            Xs, W, Y, unit, cyc, Fc, hs, T = d
            blocks.append((Xs, W, Y, unit + off, cyc, Fc, hs, T)); used.append(f"{ds.split('_')[1].split('.')[0]}/{s}")
    if not blocks: raise FileNotFoundError("N-CMAPSS 서브셋 못 찾음. DATA_DIR 확인.")
    cat = lambda j: np.concatenate([b[j] for b in blocks])
    Xs, W, Y, U, C, Fc, H = (cat(j) for j in range(7))
    T = np.concatenate([b[7] for b in blocks]) if all(b[7] is not None for b in blocks) else None
    HAS_THETA = T is not None
    flab = _fault_per_unit(T, U, H); FL = np.array([flab[u] for u in U], int)   # ← 2단계
    THETA = _theta_dev(T) if T is not None else np.zeros((len(U), NMOD), np.float32)  # ← 2단계
    print(f"  [데이터] {used} · 유닛 {len(set(U.tolist()))} · 고장모듈 {'O' if T is not None else 'X'} · θ건강지표 {'O' if HAS_THETA else 'X'}")
    return (Xs, W, Y, U, C, Fc, H, FL, THETA), (T.shape[1] // 2 if T is not None else 1)

print("✅ 1단계 정의 완료 (load_multi)")

## 2단계 — θ로 정답지(고장 부위 / 닳음) 만들기

**목표**: 1단계에서 읽은 θ를 **학습에 쓸 정답 라벨**로 가공합니다. θ 자체는 각 모듈의 효율·유량 파라미터인데, **θ=1이면 정상, 1에서 멀어질수록 열화**입니다.

**핵심 함수**
- `_theta_dev(T)` — θ를 **모듈별 열화량 [N, 5]** 로 바꿉니다. 각 모듈의 두 파라미터에서 `|θ-1|`을 구해 평균 → 0이면 정상, 클수록 닳음. 이게 **④건강지표 회귀의 정답**입니다.
- `_fault_per_unit(T, unit, hs)` — 엔진마다 **어느 모듈이 고장인지** 한 개의 라벨을 만듭니다. 수명 끝 무렵(`-k`)의 θ 편차를 건강할 때(`hs==1`)와 비교해, 가장 많이 닳은 모듈을 argmax로 고릅니다. 이게 **②고장 부위 분류의 정답**입니다.

**주의(정직)**: θ는 시뮬레이터가 아는 값이라 실엔진엔 없습니다. 그래서 여기서 만든 라벨은 "학습·평가용"이고, 실전 추론은 센서로만 합니다.

이 두 함수는 1단계 `load_multi` 안에서 호출되지만, 개념상 2단계라 따로 정의합니다. (정의 순서는 상관없고, `run()` 호출 전에 둘 다 정의돼 있으면 됩니다.)

In [ ]:
# ===== [2단계] θ → 라벨(고장모듈 / 모듈별 닳음) =====
def _theta_dev(T):
    """θ → 모듈별 열화량 [N, NMOD] = |θ-1|의 모듈쌍 평균 (0=정상, 값↑=열화)."""
    nmod = min(T.shape[1] // 2, NMOD)
    out = np.zeros((len(T), NMOD), np.float32)
    for j in range(nmod):
        out[:, j] = np.abs(T[:, 2 * j:2 * j + 2] - 1.0).mean(1)
    return out

def _fault_per_unit(T, unit, hs):
    """엔진마다 '가장 많이 닳은 모듈' 1개를 고장 라벨로. (수명 끝 θ 편차 argmax)"""
    if T is None: return {u: 0 for u in np.unique(unit)}
    nmod = T.shape[1] // 2; lab = {}
    for u in np.unique(unit):
        m = unit == u; Tu = T[m]; hu = hs[m]; k = max(1, len(Tu) // 10)
        healthy = Tu[hu == 1].mean(0) if (hu == 1).sum() > 0 else Tu[:k].mean(0)
        dev = np.abs(Tu[-k:].mean(0) - healthy)
        mod = [dev[2 * j:2 * j + 2].sum() for j in range(nmod)]
        lab[u] = int(np.argmax(mod)) if nmod > 0 else 0
    return lab

print("✅ 2단계 정의 완료 (θ 라벨)")

## 3단계 — 특징 다듬기 (regime 정규화 + 잔차 + 윈도우)

**목표**: 센서값을 모델에 넣기 전에 **"운항 상황 효과"를 제거**합니다. 엔진은 고도·속도가 제각각인 여러 조건에서 날기 때문에, 그대로 두면 "값이 높은 게 상황 때문인지 고장 때문인지" 구분이 안 됩니다.

**핵심 아이디어 두 가지**
- **Regime 정규화**: 운항조건 `W`를 KMeans로 **8개 regime(상황 그룹)** 으로 묶고, 센서를 **그룹별로 표준화**합니다. → "이 상황에선 원래 이 정도" 기준으로 보정.
- **잔차(residual)**: 건강한 기준선 대비 얼마나 벗어났는지를 **추가 특징**으로 붙입니다.

**핵심 함수**
- `make_fold_data(pool, train_units, eval_units)` — **여기가 누수 차단의 핵심**입니다. regime 군집(KMeans), 정규화 통계(평균·표준편차), 최종 표준화까지 **전부 train 유닛에서만 fit**하고 eval엔 적용만 합니다(우리가 내내 지킨 원칙). 최종 입력은 [regime정규화 센서 + 잔차 + 정규화된 운항조건] 세 묶음을 이어붙인 것입니다.
- `_windows(...)` — 유닛·사이클 경계로 나눈 뒤 `DOWN`으로 솎고 길이 `L` 창을 `STRIDE`씩 밀어 윈도우를 만듭니다. 각 창의 끝 시점에서 RUL(상한 CAP 적용)·고장·hs·θ 라벨을 답니다.
- `HierBaseline` — 계층 베이즈 기준선(옵션, `USE_HIER=False`로 기본 꺼짐).

**현재 코드와의 차이**: 기존엔 윈도우에서 mean·std 같은 통계량을 뽑아 트리에 넣고 MinMax만 했어요. 여기선 상황 보정(regime)+잔차를 거친 시퀀스를 그대로 CNN에 넣습니다. 누수 차단(폴드 내 train fit)은 동일합니다.

In [ ]:
# ===== [3단계] regime 정규화 + 잔차 + 윈도우 =====
def _regimes(W, w_mu, w_sd, km):
    return np.zeros(len(W), int) if km is None else km.predict((W - w_mu) / w_sd)

class HierBaseline:
    """계층 베이즈 건강 기준선 (옵션; USE_HIER=False면 안 씀)."""
    def fit(self, Xs, reg, unit, healthy, n_reg):
        Cc = Xs.shape[1]; self.h_pop = np.zeros((n_reg, Cc), np.float32)
        self.tau2 = np.full((n_reg, Cc), 1e-6, np.float32); self.s2 = np.ones((n_reg, Cc), np.float32)
        for r in range(n_reg):
            hm = (reg == r) & (healthy == 1); base = Xs[hm] if hm.sum() > 0 else (Xs[reg == r] if (reg == r).sum() > 0 else Xs); self.h_pop[r] = base.mean(0)
            means, wv, ns = [], [], []
            for u in np.unique(unit):
                m = hm & (unit == u)
                if m.sum() >= 2: means.append(Xs[m].mean(0)); wv.append(Xs[m].var(0)); ns.append(int(m.sum()))
            if len(means) >= 2:
                self.s2[r] = np.maximum(np.mean(wv, 0), 1e-6); self.tau2[r] = np.maximum(np.stack(means).var(0) - self.s2[r] / max(np.mean(ns), 1.0), 1e-6)
        return self
    def unit_h(self, Xu, ru, hu, n_reg):
        h = self.h_pop.copy()
        for r in range(n_reg):
            m = (ru == r) & (hu == 1); n = int(m.sum())
            if n >= 1: w = self.tau2[r] / (self.tau2[r] + self.s2[r] / max(n, 1)); h[r] = w * Xu[m].mean(0) + (1 - w) * self.h_pop[r]
        return h

def _windows(feats, Y, unit, cyc, Fc, FL, HS, TH):
    """유닛·사이클 경계로 나눠 DOWN 솎기 → 길이 L 창을 STRIDE씩. 끝 시점 라벨 부착."""
    chg = np.where((np.diff(unit) != 0) | (np.diff(cyc) != 0))[0] + 1
    Xw, Yw, Fw, Uw, FLw, HSw, THw = [], [], [], [], [], [], []
    for idx in np.split(np.arange(len(unit)), chg):
        if len(idx) < DOWN * L: continue
        ds = idx[::DOWN]
        for s in range(0, len(ds) - L + 1, STRIDE):
            win = ds[s:s + L]; e = win[-1]
            Xw.append(feats[win]); Yw.append(min(Y[e], CAP)); Fw.append(int(Fc[e])); Uw.append(int(unit[e]))
            FLw.append(int(FL[e])); HSw.append(int(HS[e])); THw.append(TH[e])
    return (np.asarray(Xw, np.float32), np.asarray(Yw, np.float32), np.asarray(Fw, int), np.asarray(Uw, int),
            np.asarray(FLw, int), np.asarray(HSw, int), np.asarray(THw, np.float32))

def make_fold_data(pool, train_units, eval_units):
    """★ 누수 차단: regime·정규화·표준화 통계를 전부 train 유닛에서만 fit."""
    Xs, W, Y, U, C, F, H, FL, TH = pool
    trm = np.isin(U, train_units); evm = np.isin(U, eval_units)
    w_mu = W[trm].mean(0); w_sd = W[trm].std(0) + 1e-6
    km = MiniBatchKMeans(N_REGIMES, random_state=0, n_init=3, batch_size=4096).fit((W[trm] - w_mu) / w_sd) if (HAVE_SK and N_REGIMES > 1) else None
    nreg = N_REGIMES if km is not None else 1; reg_all = _regimes(W, w_mu, w_sd, km)
    rmu = np.zeros((nreg, Xs.shape[1]), np.float32); rsd = np.ones((nreg, Xs.shape[1]), np.float32)
    for r in range(nreg):
        m = (reg_all == r) & trm
        if m.sum() >= 2: rmu[r] = Xs[m].mean(0); rsd[r] = Xs[m].std(0) + 1e-6
    hb = HierBaseline().fit(Xs[trm], reg_all[trm], U[trm], H[trm], nreg) if USE_HIER else None
    tr_smu = Xs[trm].mean(0); tr_ssd = Xs[trm].std(0) + 1e-6

    def feats_for(mask):
        out_idx = np.where(mask)[0]
        z = ((Xs[mask] - rmu[reg_all[mask]]) / rsd[reg_all[mask]]) if USE_REGIME_NORM else (Xs[mask] - tr_smu) / tr_ssd
        parts = [z]; res = np.zeros_like(Xs[mask])
        if USE_RESIDUAL:
            if USE_HIER:
                for u in np.unique(U[mask]):
                    um = U[mask] == u; gi = out_idx[um]; h = hb.unit_h(Xs[gi], reg_all[gi], H[gi], nreg); res[um] = (Xs[gi] - h[reg_all[gi]]) / rsd[reg_all[gi]]
            else:
                for r in range(nreg):
                    rm = reg_all[mask] == r
                    if rm.any(): res[rm] = (Xs[mask][rm] - rmu[r]) / rsd[r]
            parts.append(res)
        parts.append((W[mask] - w_mu) / w_sd)
        return np.concatenate(parts, 1).astype(np.float32)

    tr = _windows(feats_for(trm), Y[trm], U[trm], C[trm], F[trm], FL[trm], H[trm], TH[trm])
    ev = _windows(feats_for(evm), Y[evm], U[evm], C[evm], F[evm], FL[evm], H[evm], TH[evm])
    if len(tr[0]) > MAX_WIN:
        sel = rng.choice(len(tr[0]), MAX_WIN, replace=False); tr = tuple(a[sel] for a in tr)
    mu = tr[0].reshape(-1, tr[0].shape[-1]).mean(0); sd = tr[0].reshape(-1, tr[0].shape[-1]).std(0) + 1e-6
    nm = lambda A: ((A - mu) / sd).astype(np.float32)
    return (nm(tr[0]),) + tr[1:], (nm(ev[0]),) + ev[1:]

print("✅ 3단계 정의 완료 (make_fold_data)")

## 4단계 — 두뇌 교체: 멀티헤드 CNN

**목표**: 트리·LSTM을 따로 쓰는 대신, **하나의 CNN이 네 가지를 동시에** 예측하게 합니다. 여기가 가장 큰 구조 변화입니다.

**구조**: Conv1d 3층(커널 7→5→3, 채널 32→64→64, 각 BatchNorm·ReLU·Dropout) → `AdaptiveAvgPool`로 시간축 압축 → 공유 head(Linear 64→64) → **5개 출력 머리**로 분기.

**5개 머리**
- `mu` — RUL 평균(예측값)
- `lv` — RUL 로그분산(=얼마나 불확실한지, 5단계에서 사용)
- `fault` — 고장 모듈 분류(클래스별 점수)
- `hsh` — 열화 시작 여부(0/1)
- `theta` — 모듈별 닳음 회귀(softplus로 0 이상)

**왜 한 번에?** 여러 작업을 같이 배우면(멀티태스크) 공유 CNN이 더 일반적인 특징을 학습해 서로 도움을 줄 수 있고, 무엇보다 **한 번 학습으로 네 결과가 다 나옵니다**.

In [ ]:
# ===== [4단계] 멀티헤드 CNN (RUL·고장·hs·θ) =====
class MultiCNN(nn.Module):
    def __init__(self, C, n_fault, n_theta):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(C, 32, 7, padding=3), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.1),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.1),
            nn.Conv1d(64, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(), nn.AdaptiveAvgPool1d(1))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.2))
        self.mu = nn.Linear(64, 1); self.lv = nn.Linear(64, 1)          # RUL 평균 / 로그분산
        self.fault = nn.Linear(64, max(n_fault, 1)); self.hsh = nn.Linear(64, 1)  # 고장 / 열화시작
        self.theta = nn.Linear(64, n_theta)                            # 건강지표(모듈별 닳음)
    def forward(self, x):
        z = self.head(self.net(x.transpose(1, 2)))
        return (self.mu(z).squeeze(-1), self.lv(z).squeeze(-1).clamp(-6, 6),
                self.fault(z), self.hsh(z).squeeze(-1), Fn.softplus(self.theta(z)))   # θ≥0

print("✅ 4단계 정의 완료 (MultiCNN)")

## 5단계 — 불확실성 + 딥앙상블

**목표**: 예측에 **신뢰도**를 붙이고, 모델 여러 개를 평균내 **결과를 안정화**합니다.

**불확실성 (heteroscedastic NLL)**: 모델이 평균 `mu`만이 아니라 **분산 `lv`도 함께** 예측하게 합니다. 처음 `WARMUP_MSE`(5) 에폭은 단순 MSE로 안정화한 뒤, 그다음부터 **가우시안 음의로그가능도(NLL)** 로 바꿔서 `rul_ps = 0.5*(lv + d²·exp(-lv))` 를 최소화합니다. 헷갈리는 구간은 "분산 크다"고 표현하게 되는 거예요.

**멀티태스크 손실**: RUL 손실에 **고장 분류(cross-entropy) + 열화시작(BCE) + θ 회귀(MSE)** 를 `LAMBDA_*` 가중으로 더합니다. (6단계의 클래스 가중치도 여기 `train_one` 안에 들어 있습니다.)

**딥앙상블**: fold마다 **시드가 다른 모델 `ENSEMBLE_M`(3)개**를 학습해 예측을 평균냅니다(`ensemble_predict`). 모델마다 조금씩 다르게 틀리므로, 평균내면 분산이 줄어 더 안정적입니다.

**핵심 함수**: `train_one`(한 모델 학습), `_predict`(한 모델 예측), `ensemble_predict`(M개 평균).

In [ ]:
# ===== [5단계] 학습(불확실성 NLL) + 딥앙상블 =====  ([6단계] 클래스 가중치도 여기 포함)
def term(Ls, t):
    if t <= 0: return Ls.mean()
    return (torch.logsumexp(t * Ls, 0) - np.log(Ls.shape[0])) / t

def _loader(arrs, sh=False):
    return DataLoader(TensorDataset(*[torch.tensor(a) for a in arrs]), batch_size=BATCH, shuffle=sh)
def rmse(a, b): return float(np.sqrt(((a - b) ** 2).mean())) if len(a) else float('nan')

def train_one(seed, tr, n_fault):
    Xtr, Ytr, Ftr, Utr, FLtr, HStr, THtr = tr
    torch.manual_seed(seed); np.random.seed(seed)
    model = MultiCNN(Xtr.shape[-1], n_fault, NMOD).to(dev)
    fault_w = None
    # ── [6단계] 고장분류 붕괴 방지: 흔한 클래스는 가중치↓, 드문 클래스↑ ──
    if MULTITASK and FAULT_CLASS_WEIGHT:
        _cnt = np.bincount(FLtr, minlength=max(n_fault, 1)).astype(np.float32); _cnt[_cnt == 0] = 1.0
        _w = _cnt.sum() / (len(_cnt) * _cnt); fault_w = torch.tensor(_w, dtype=torch.float32).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    for ep in range(EPOCHS):
        model.train(); use_nll = ep >= WARMUP_MSE      # 워밍업 후 불확실성(NLL) 켜기
        for xb, yb, fb, ub, flb, hsb, thb in _loader((Xtr, Ytr, Ftr, Utr, FLtr, HStr, THtr), sh=True):
            xb, yb, flb, hsb, thb = xb.to(dev), yb.to(dev), flb.to(dev), hsb.float().to(dev), thb.to(dev)
            opt.zero_grad(); mu, lv, fault, hsl, th = model(xb); d = mu - yb
            rul_ps = (0.5 * (lv + d ** 2 * torch.exp(-lv))) if use_nll else (d ** 2)
            loss = term(rul_ps, TERM_T)
            if MULTITASK:
                loss = loss + LAMBDA_FAULT * Fn.cross_entropy(fault, flb, weight=fault_w) + LAMBDA_HS * Fn.binary_cross_entropy_with_logits(hsl, hsb)
                if HAS_THETA and LAMBDA_THETA > 0: loss = loss + LAMBDA_THETA * Fn.mse_loss(th, thb)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
        sch.step()
    return model

def _predict(model, ev):
    model.eval(); mus, fas, hss, ths = [], [], [], []
    with torch.no_grad():
        for xb, *_ in _loader(ev):
            mu, lv, fault, hsl, th = model(xb.to(dev)); mus.append(mu.cpu().numpy())
            fas.append(torch.softmax(fault, 1).cpu().numpy()); hss.append(torch.sigmoid(hsl).cpu().numpy()); ths.append(th.cpu().numpy())
    return np.concatenate(mus), np.concatenate(fas), np.concatenate(hss), np.concatenate(ths)

def ensemble_predict(models, ev):
    M = [_predict(m, ev) for m in models]
    mu = np.clip(np.stack([x[0] for x in M]).mean(0), 0, CAP)
    fa = np.stack([x[1] for x in M]).mean(0); hs = np.stack([x[2] for x in M]).mean(0); th = np.stack([x[3] for x in M]).mean(0)
    return mu, fa, hs, th

print("✅ 5단계 정의 완료 (train_one · ensemble_predict)")

## 6단계 — 고장 분류 살리기 (v12의 핵심 목표)

**문제**: 고장 중 특정 모듈(예: LPT)이 워낙 흔해서, 모델이 **항상 그 모듈만 찍어버리는 붕괴**가 일어납니다. 그러면 정확도가 "그냥 제일 흔한 답의 비율"(예: 0.618)에 머물러요 — 아무것도 안 배운 거죠.

**세 가지 처방** (이 단계의 코드는 5단계 `train_one`과 8단계 `run`에 나눠 들어 있습니다):
1. **분류헤드 + 클래스 가중치** — `train_one`에서 흔한 클래스의 손실 가중치를 낮춰(`fault_w`), "흔한 답만 외치면 손해"가 되게 합니다. 유닛별 다수결로 최종 고장을 정합니다.
2. **θ raw argmax** — 예측한 모듈별 닳음 중 가장 큰 모듈을 고장으로.
3. **θ z-정규화 argmax** — 모듈마다 값 크기 편향이 있어서, 모듈별로 z-정규화해 편향을 제거한 뒤 argmax.

세 방법의 유닛 정확도를 **majority-baseline(=dumb 0.618)** 과 나란히 찍고, **BEST가 baseline을 이기면 분류가 실제로 살아난 것**입니다. `run()`이 이 비교를 출력합니다.

## 7단계 — 예측구간 (conformal)

**목표**: "RUL 30" 같은 점 예측을 넘어, **"RUL 30 ± Q"처럼 커버리지가 보장된 구간**을 만듭니다.

**원리**: 검증 잔차(`|실제-예측|`)를 모아 `(1-ALPHA)`(90%) 분위수 `Q`를 구하면, "이 정도 오차 안에 90%가 든다"는 안전선이 됩니다. 비행클래스별로 조건부(`GROUP_CONDITIONAL`) 적용하고, 표본이 적은 그룹은 전체 분위수로 대체합니다.

**핵심 함수**: `conformal_Q(res, F, alpha, group)`. 그리고 `run()`에서 **fold-out**(그 fold를 뺀 나머지)로 Q를 구해 해당 fold에 적용하는 cross-conformal 방식으로 **실제 커버리지**(`covmarg`)를 측정합니다 — 목표 90%에 근접하는지 확인용입니다.

In [ ]:
# ===== [7단계] conformal 예측구간 =====
def conformal_Q(res, F, alpha, group):
    """검증 잔차의 (1-alpha) 분위수 Q. 비행클래스별 조건부(옵션)."""
    def q_of(idx):
        if len(idx) == 0: return 0.0
        n = len(idx); k = min(int(np.ceil((n + 1) * (1 - alpha))), n); return float(np.sort(res[idx])[k - 1])
    if not group: return {c: q_of(np.arange(len(res))) for c in (1, 2, 3)}
    return {c: (q_of(np.where(F == c)[0]) if (F == c).sum() >= MIN_CAL_PER_GROUP else q_of(np.arange(len(res)))) for c in (1, 2, 3)}

print("✅ 7단계 정의 완료 (conformal_Q)")

## 8단계 — 체크포인트 방식 실행 (fold별 저장/재개)

`run()` 하나로 다 돌리는 대신, 다음 3개로 나눕니다:

- **`prepare()`** — 데이터를 읽고(1·2단계) 유닛을 고정 시드로 fold로 나눕니다. 이 분할은 `meta.pkl`에 저장돼, 다음 날에도 **똑같은 fold 구성**이 재현됩니다.
- **`run_fold(k, meta)`** — fold k를 처리합니다. 이미 `fold_k_preds.pkl`이 있으면 **예측만 불러오고 학습을 건너뜁니다.** 없으면 모델 M개를 학습하는데, **모델 하나씩 저장**(`fold_k_model_i.pt`)해서 중간에 끊겨도 이어집니다. 끝나면 그 fold의 예측을 저장합니다.
- **`finalize(meta)`** — 모든 fold의 예측을 모아 최종 지표(RUL·고장 3방법·hs·θ·커버리지)를 계산하고 JSON으로 저장합니다. (모든 fold가 끝나야 실행됨.)

**다음 날 재개 흐름**: 정의 셀들 실행 → `prepare()`(meta 있으면 즉시) → fold 루프(끝난 fold는 몇 초 만에 스킵, 안 끝난 것만 학습) → `finalize()`. 학습을 처음부터 다시 하지 않습니다.

**주의**: 재개할 때 CONFIG(특히 `L·DOWN·STRIDE·USE_RESIDUAL` 등 특징 관련)를 바꾸면 저장된 모델과 입력 크기가 안 맞아 로드가 실패할 수 있어요. 같은 CAP 실험은 **CONFIG를 그대로** 두고 이어가세요.

In [ ]:
# ===== [8단계] 체크포인트 방식 실행부 =====
_POOL = None; _NMOD = None
def _get_pool():
    """pool을 한 번만 로드해 재사용(같은 세션)."""
    global _POOL, _NMOD
    if _POOL is None:
        _POOL, _NMOD = load_multi()
    return _POOL, _NMOD

def prepare():
    """pool 로드 + fold 분할. meta.pkl 캐시(재개 시 동일 fold 재현)."""
    meta_path = f"{CKPT_DIR}/meta.pkl"
    if os.path.exists(meta_path):
        meta = pickle.load(open(meta_path, 'rb'))
        print(f"[prepare] meta 불러옴 · 유닛 {len(meta['units'])} · {len(meta['folds'])}-fold (pool은 필요 시 로드)")
        return meta
    pool, nmod = _get_pool()
    U_all = pool[3]
    units = np.array(sorted(set(U_all.tolist())))
    r = np.random.RandomState(SEED0); r.shuffle(units)                  # 고정 시드 → 재현
    folds = [f.tolist() for f in np.array_split(units, min(K_FOLDS, len(units)))]
    meta = {'units': units.tolist(), 'folds': folds, 'nmod': int(nmod)}
    pickle.dump(meta, open(meta_path, 'wb'))
    print(f"[prepare] 새로 생성 · 유닛 {len(units)} · {len(folds)}-fold · θ모듈 {nmod} → meta.pkl 저장")
    return meta

def run_fold(k, meta):
    """fold k: 완료면 예측 로드(스킵). 아니면 모델 하나씩 학습·저장 후 예측 저장."""
    pred_path = f"{CKPT_DIR}/fold_{k}_preds.pkl"
    if os.path.exists(pred_path):
        print(f"  fold {k}: ✅ 완료됨(예측 로드) — 학습 스킵")
        return pickle.load(open(pred_path, 'rb'))
    units = np.array(meta['units']); tu = np.array(meta['folds'][k]); nmod = meta['nmod']
    pool, _ = _get_pool()
    print(f"  fold {k}: 데이터 준비 중(regime·윈도우)...")
    tr, ev = make_fold_data(pool, units[~np.isin(units, tu)], tu)
    C = tr[0].shape[-1]
    models = []
    for i in range(ENSEMBLE_M):
        mpath = f"{CKPT_DIR}/fold_{k}_model_{i}.pt"
        if os.path.exists(mpath):
            m = MultiCNN(C, nmod, NMOD).to(dev)
            m.load_state_dict(torch.load(mpath, map_location=dev)['state_dict'])
            print(f"    model {i+1}/{ENSEMBLE_M}: 로드(스킵)")
        else:
            print(f"    model {i+1}/{ENSEMBLE_M}: 학습 중...")
            m = train_one(SEED0 + 10 * k + i, tr, nmod)
            torch.save({'state_dict': m.state_dict(), 'C': int(C), 'n_fault': int(nmod)}, mpath)  # ★ 모델 저장
        models.append(m)
    mu, fa, hsp, thp = ensemble_predict(models, ev)
    Xe, Ye, Fe, Ue, FLe, HSe, THe = ev
    P = {'mu': mu, 'y': Ye, 'F': Fe, 'U': Ue, 'res': np.abs(Ye - mu), 'fold': np.full(len(mu), k, int),
         'fa': fa.argmax(1), 'fl': FLe, 'hs': HSe, 'hsp': (hsp > 0.5).astype(int), 'thp': thp, 'tht': THe}
    pickle.dump(P, open(pred_path, 'wb'))                               # ★ 예측(체크포인트) 저장
    print(f"  fold {k}: 💾 저장 완료 (win {len(mu)})")
    return P

def fold_status(meta):
    """어느 fold가 끝났는지 표시."""
    done = []
    for k in range(len(meta['folds'])):
        ok = os.path.exists(f"{CKPT_DIR}/fold_{k}_preds.pkl")
        nm = sum(os.path.exists(f"{CKPT_DIR}/fold_{k}_model_{i}.pt") for i in range(ENSEMBLE_M))
        done.append(ok); print(f"  fold {k}: {'✅ 완료' if ok else f'⬜ 미완(모델 {nm}/{ENSEMBLE_M})'}")
    print(f"→ {sum(done)}/{len(done)} fold 완료")
    return all(done)

def finalize(meta):
    """모든 fold 예측을 모아 최종 지표 계산 + JSON 저장."""
    Ps = []
    for k in range(len(meta['folds'])):
        p = f"{CKPT_DIR}/fold_{k}_preds.pkl"
        if not os.path.exists(p):
            print(f"⚠️ fold {k} 아직 미완 → finalize 불가. run_fold({k}, meta) 먼저 실행."); return None
        Ps.append(pickle.load(open(p, 'rb')))
    O = {key: np.concatenate([P[key] for P in Ps]) for key in Ps[0]}
    mu, y, F, Uw, res, fold = (O[k] for k in ['mu', 'y', 'F', 'U', 'res', 'fold'])
    nfold = len(meta['folds'])

    uinfo = {u: (int(F[Uw == u][0]), rmse(mu[Uw == u], y[Uw == u])) for u in sorted(set(Uw.tolist()))}
    per = {}
    for c in (1, 2, 3):
        v = [r for (cl, r) in uinfo.values() if cl == c]
        per[FCN[c]] = ([round(float(np.mean(v)), 2), round(float(np.std(v)), 2), len(v)] if v else [float('nan'), 0, 0])
    allr = [r for _, r in uinfo.values()]
    overall = [round(rmse(mu, y), 2), round(float(np.mean(allr)), 2), round(float(np.std(allr)), 2)]

    cov = np.zeros(len(mu), bool)                                       # [7단계] conformal 커버리지
    for k in range(nfold):
        te = fold == k
        Q = conformal_Q(res[fold != k], F[fold != k], ALPHA, GROUP_CONDITIONAL)
        cov[te] = np.abs(y[te] - mu[te]) <= np.array([Q[c] for c in F[te]])
    covmarg = round(float(cov.mean()), 3)

    fault_acc = hs_acc = None; fault_dist = {}; fault_permod = {}; theta_rmse = None; theta_permod = {}
    fault_acc_thr = fault_acc_thz = None; fault_permod_thr = {}; fault_permod_thz = {}
    if MULTITASK:                                                       # [6단계] 고장 3방법 + hs + θ
        from collections import Counter as _C
        fa_pred, fl_true, hsp, hs_true = O['fa'], O['fl'], O['hsp'], O['hs']
        utrue = {u: int(fl_true[Uw == u][0]) for u in set(Uw.tolist())}
        upred = {u: int(np.bincount(fa_pred[Uw == u]).argmax()) for u in set(Uw.tolist())}
        def _acc(up): return round(sum(up[u] == utrue[u] for u in utrue) / max(len(utrue), 1), 3)
        def _pm(up):
            d = {}
            for c in range(NMOD):
                us = [u for u in utrue if utrue[u] == c]
                if us: d[MODULES[c]] = round(sum(up[u] == c for u in us) / len(us), 2)
            return d
        hs_acc = round(float((hsp == hs_true).mean()), 3)
        cnt = _C(utrue.values()); fault_dist = {MODULES[c]: cnt[c] for c in range(NMOD) if cnt.get(c, 0) > 0}
        active = sorted(set(utrue.values()))
        fault_acc = _acc(upred); fault_permod = _pm(upred)             # (1) 분류헤드(클래스가중)
        if HAS_THETA:
            thp_all = O['thp']
            uth = {u: thp_all[Uw == u].mean(0) for u in utrue}
            def _amx(vec): return active[int(np.argmax([vec[c] for c in active]))]
            upred_thr = {u: _amx(uth[u]) for u in uth}                 # (2) θ raw argmax
            UU = list(uth.keys()); Marr = np.stack([uth[u] for u in UU])
            Mz = (Marr - Marr.mean(0)) / (Marr.std(0) + 1e-9)
            upred_thz = {UU[i]: active[int(np.argmax([Mz[i, c] for c in active]))] for i in range(len(UU))}  # (3) θ z argmax
            fault_acc_thr = _acc(upred_thr); fault_permod_thr = _pm(upred_thr)
            fault_acc_thz = _acc(upred_thz); fault_permod_thz = _pm(upred_thz)
            thp, tht = O['thp'], O['tht']
            theta_rmse = round(float(np.sqrt(((thp - tht) ** 2).mean())), 4)
            theta_permod = {MODULES[j]: round(float(np.sqrt(((thp[:, j] - tht[:, j]) ** 2).mean())), 4) for j in range(NMOD)}

    print(f"\n===== RUL v12 (CAP={CAP} · 유닛교차 {nfold}fold · M{ENSEMBLE_M}) =====")
    print(f"  [RUL] 전체 pool {overall[0]} / 유닛 {overall[1]}±{overall[2]} · 커버리지 {covmarg}")
    print(f"  [RUL] 비행클래스별 {per}")
    if MULTITASK:
        print(f"  [고장 부위] 유닛정확도 {fault_acc} · 모듈별 {fault_permod} · 분포 {fault_dist}")
        print(f"  [열화 시작] hs 정확도 {hs_acc}")
        if HAS_THETA:
            print(f"  [건강지표 θ] 전체 RMSE {theta_rmse} (0=정상) · 모듈별 {theta_permod}")
    _base = round(max(fault_dist.values()) / sum(fault_dist.values()), 3) if (MULTITASK and fault_dist) else None
    if MULTITASK and fault_dist:
        print(f"  [fault] majority-baseline(dumb) {_base} | clf(weighted) {fault_acc}")
        if HAS_THETA:
            print(f"  [fault] theta-raw {fault_acc_thr} · theta-z {fault_acc_thz}")
            _cands = [("clf", fault_acc), ("theta_raw", fault_acc_thr), ("theta_z", fault_acc_thz)]
            _best = max(_cands, key=lambda x: x[1]); print(f"  [fault] BEST = {_best[0]} ({_best[1]}) vs baseline {_base}")

    out_path = f"{RESULT_DIR}/results_rul_v12_cap{CAP}.json"            # [8단계] 저장
    json.dump({"model": "v12-8stage", "CAP": CAP, "L": L, "DOWN": DOWN, "STRIDE": STRIDE,
               "n_units": len(meta['units']), "n_folds": nfold,
               "rmse_overall_pool": overall[0], "rmse_overall_unit_mean": overall[1], "rmse_overall_unit_std": overall[2],
               "rmse_by_class": per, "coverage_marginal": covmarg,
               "fault_majority_baseline": _base, "fault_distribution": fault_dist,
               "fault_clf_accuracy": fault_acc, "fault_clf_per_module": fault_permod,
               "fault_theta_raw_accuracy": fault_acc_thr, "fault_theta_raw_per_module": fault_permod_thr,
               "fault_theta_z_accuracy": fault_acc_thz, "fault_theta_z_per_module": fault_permod_thz,
               "hs_accuracy": hs_acc, "theta_rmse": theta_rmse, "theta_per_module": theta_permod},
              open(out_path, "w"), ensure_ascii=False, indent=2)
    print(f"\n저장: {out_path}")
    return overall, fault_acc, theta_rmse

print("✅ 8단계(체크포인트 실행부) 정의 완료 — prepare / run_fold / fold_status / finalize")

## 실행 방법

**① 준비** — 데이터를 읽고 fold를 나눕니다(처음 한 번은 pool 로드 때문에 몇 분).

**② 상태 확인**(선택) — 어느 fold까지 끝났는지 봅니다.

**③ 전체 실행(재개 가능)** — fold를 순서대로 처리하되, 이미 끝난 fold는 즉시 스킵합니다. **이 셀을 다시 실행하면** 멈춘 지점부터 이어집니다. 끊겨도 이 셀만 다시 누르면 됩니다.

**하루에 하나씩** 돌리고 싶으면 `run_fold(0, meta)`, 다음 날 `run_fold(1, meta)` … 처럼 **fold별 셀**을 하나씩 실행해도 됩니다(아래 예시).

**④ 최종 집계** — 모든 fold가 끝나면 지표를 계산하고 JSON으로 저장합니다.

In [ ]:
# ===== ① 준비 (pool 로드 + fold 분할) =====
meta = prepare()

In [ ]:
# ===== ② 상태 확인 (선택) =====
all_done = fold_status(meta)

In [ ]:
# ===== ③ 전체 실행 (재개 가능 — 끊기면 이 셀만 다시 실행) =====
for k in range(len(meta['folds'])):
    run_fold(k, meta)
print("\n모든 fold 처리 시도 완료. 다음 셀에서 finalize() 하세요.")

In [ ]:
# ===== (선택) 하루에 하나씩: fold 개별 실행 =====
# 필요한 fold 번호만 주석 해제해서 실행하세요. 이미 끝난 fold는 즉시 스킵됩니다.
run_fold(0, meta)
# run_fold(1, meta)
# run_fold(2, meta)
# run_fold(3, meta)
# run_fold(4, meta)
# run_fold(5, meta)

In [ ]:
# ===== ④ 최종 집계 + 저장 (모든 fold 끝난 뒤) =====
result = finalize(meta)

## (선택) 저장된 모델 불러와 예측하기

각 fold의 최종 모델은 `fold_k_model_i.pt`로 저장돼 있습니다. 나중에 새 데이터로 예측할 때 아래처럼 불러옵니다. (입력 채널 수 `C`는 저장 시 함께 기록돼 있어, 같은 CONFIG로 만든 특징이면 그대로 맞습니다.)

In [ ]:
# 예시: fold 0의 모델 3개를 불러와 앙상블 예측 (새 데이터 준비는 make_fold_data 필요)
# ck = torch.load(f"{CKPT_DIR}/fold_0_model_0.pt", map_location=dev)
# m = MultiCNN(ck['C'], ck['n_fault'], NMOD).to(dev)
# m.load_state_dict(ck['state_dict']); m.eval()
# mu, fa, hs, th = ensemble_predict([m], ev_new)   # ev_new: make_fold_data로 만든 평가 윈도우
print("저장된 모델 경로 예시:", f"{CKPT_DIR}/fold_0_model_0.pt")